# **1.Exploratory Data Analysis**

*Загрузка датасета*

**1.1 Общие сведения о датасете**

In [ ]:
from MyDB import MyDB
import pandas as pd
import numpy as np
from typing import Tuple, List, Literal, Dict

def prepare_dataframe(df: pd.DataFrame, in_place: bool = False) -> pd.DataFrame:
    for column in df.columns:
        match (column):
            case "job_title":
                continue
            
            case "experience_years":
                df["experience_years"] = df["experience_years"].astype(dtype = "int", copy = in_place)
                continue
                
            case "education_level":
                df["education_level"] = df["education_level"].astype(dtype = "category", copy = in_place)
                continue
                
            case "skills_count":
                df["skills_count"] = df["skills_count"].astype(dtype = "int", copy = in_place)
                continue
                
            case "industry":
                df["industry"] = df["industry"].astype(dtype = "category", copy = in_place)
                continue
                
            case "company_size":
                df["company_size"] = df["company_size"].astype(dtype = "category", copy = in_place)
                continue
            
            case "location":
                continue
            
            case "remote_work":
                df["remote_work"] = df["remote_work"].astype(dtype = "category", copy = in_place)
                continue
                
            case "certifications":
                df["certifications"] = df["certifications"].astype(dtype = "int", copy = in_place)
                continue
                
            case "salary":
                df["salary"] = df["salary"].astype(dtype = "float", copy = in_place)
                continue
    
    return df

def load_data(query: str = None, num: int = 1000, random: bool = False) -> pd.DataFrame:
    if query is None:
        if num == -1:
            query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries;"""                
            if random == True:
                query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries
                        ORDER BY RANDOM();"""
        else:
            query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries 
                        LIMIT {num};"""
                        
            if random == True:
                query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries
                        ORDER BY RANDOM()
                        LIMIT {num};"""
    
    db = MyDB()
    db.connect_to_db()
    
    df = db.select_query(query=query)               
    df = prepare_dataframe(df = df, in_place=True)
    
    db.close_db()
    
    return df

def convert_to_batches(df: pd.DataFrame = None, num: int = 1000) -> Tuple[pd.DataFrame, pd.Series]:
    if df is None:
        df = prepare_dataframe(load_data(num = num))
        
    y = df["salary"]
    
    X = df.drop(["salary"], axis = 1)
    
    return X, y
    

df = load_data()
print(df.info())


**1.2.1 Анализ признаков(Распределение)**

In [ ]:
from EDA import EDA
df = prepare_dataframe(load_data())
eda = EDA()

eda.make_analysis_of_distribution(data=df, show_quantile=True, download=False, path_to_dir="D:\\Salary-prediction\\Distribution")

**1.2.2 Матрица корреляций числовых признаков**

In [ ]:
import matplotlib.pyplot as plt

def create_heatmap_corr(matrix_corr: pd.DataFrame, title: str = "Матрица корреляций"):
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1,1,1)

    im = ax.imshow(X=matrix_corr, cmap='hot')
    ax.set_title(title)

    ax.xaxis.set_ticks(range(0,len(matrix_corr.columns)))
    ax.xaxis.set_ticklabels(matrix_corr.columns)

    ax.yaxis.set_ticks(range(0,len(matrix_corr.columns)))
    ax.yaxis.set_ticklabels(matrix_corr.columns)

    ax.tick_params(axis='x', labelrotation = 270)
    
    for i in range(len(matrix_corr.index)):        
        for j in range(len(matrix_corr.columns)):  
            value = matrix_corr.iloc[i, j]         
            ax.text(j, i, round(value, 1),
                    ha="center", va="center", color="w")

    fig.colorbar(im)
    fig.show()
    
df = prepare_dataframe(load_data())
numeric_features = ["experience_years", "skills_count", "certifications"]
create_heatmap_corr(df[numeric_features + ["salary"]].corr(method="pearson"), "Матрица корреляций - Пирсона")


**1.2.3 Матрица корреляций ординальных признаков**

In [ ]:
def convert_ordinal(data: pd.DataFrame) -> pd.DataFrame:
    edu_mapping = {
    'High School': 0,
    'Diploma': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
    }

    size_mapping = {
        'Startup': 0,
        'Small': 1,
        'Medium': 2,
        'Large': 3,
        'Enterprise': 4
    }   
        
    data["education_level"] = data["education_level"].map(edu_mapping)
    data["company_size"] = data["company_size"].map(size_mapping)
    return data 

ordinal_features = ["education_level", "company_size"]

df = prepare_dataframe(load_data())
df = convert_ordinal(df)

create_heatmap_corr(df[ordinal_features + ["salary"]].corr(method="spearman"), "Матрица корреляций - Спирмена")


# **2.Baseline модели**

*Загрузка необходимых функций*

In [ ]:
from sklearn.dummy import DummyRegressor #Глупая регрессия
from sklearn.linear_model import LinearRegression, Ridge, Lasso #Линейная регрессия
from sklearn.tree import DecisionTreeRegressor #Решающее дерево
from sklearn.ensemble import RandomForestRegressor #Случайный лес

from sklearn.base import BaseEstimator, clone
from sklearn.model_selection import GridSearchCV, KFold, train_test_split

def print_metrics(y_true: pd.Series, y_pred: pd.Series):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
    
    print(f"MAPE: {round(mean_absolute_percentage_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"MAE: {round(mean_absolute_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"MSE: {round(mean_squared_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"RMSE: {round(root_mean_squared_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"R2: {round(r2_score(y_true=y_true, y_pred=y_pred),2)}")

def cv_fit(X: pd.DataFrame, y: pd.Series, model: BaseEstimator, num_folds: int = 5) -> BaseEstimator:
    from sklearn.metrics import root_mean_squared_error
    
    folds = KFold(n_splits=num_folds, shuffle=True, random_state=47)
    best_rmse = float('inf')
    best_model = None

    for i, (idx_train, idx_test) in enumerate(folds.split(X)):
        X_train, y_train = X.iloc[idx_train, :], y.iloc[idx_train]
        X_test, y_test = X.iloc[idx_test, :], y.iloc[idx_test]

        model_fold = clone(model)
        model_fold.fit(X_train, y_train)
        y_pred = model_fold.predict(X_test)

        rmse = round(root_mean_squared_error(y_test, y_pred),2)
        if rmse < best_rmse:
            best_rmse = rmse
            best_model = model_fold

    return best_model     

def robust_scale(X:pd.DataFrame, features: Tuple) -> pd.DataFrame:
    from sklearn.preprocessing import RobustScaler
    from sklearn.model_selection import train_test_split
    
    X_scaled = X.copy()
    
    idx_train, idx_test = train_test_split(
        X.index, 
        test_size=0.3, 
        random_state=42  
    )
    
    scaler = RobustScaler()
    
    scaler.fit(X.loc[idx_train, features])
    
    X_scaled.loc[idx_train, features] = scaler.transform(X.loc[idx_train, features])
    X_scaled.loc[idx_test, features] = scaler.transform(X.loc[idx_test, features])
    
    return X_scaled

def change_category(data: pd.DataFrame):    
    if "job_title" in data.columns:
        clusters_of_job = {
        # Data & AI
        'AI Engineer': 'Data & AI',
        'Machine Learning Engineer': 'Data & AI',
        'Data Scientist': 'Data & AI',
        'Data Analyst': 'Data & AI',
        
        # Software Engineering
        'Software Engineer': 'Software Engineering',
        'Backend Developer': 'Software Engineering',
        'Frontend Developer': 'Software Engineering',
        
        # Cloud & DevOps
        'Cloud Engineer': 'Cloud & DevOps',
        'DevOps Engineer': 'Cloud & DevOps',
        
        # Product & Strategy
        'Product Manager': 'Product & Strategy',
        'Business Analyst': 'Product & Strategy',
        
        # Security
        'Cybersecurity Analyst': 'Security'
    }
        
        data["job_title"] = data["job_title"].map(clusters_of_job)
    
    if "location" in data.columns:
        clusters_of_location = { 
        #America
        'USA': 'North America',
        'Canada': 'North America',
        
        #Europe
        'UK': 'Western Europe',
        'Germany': 'Western Europe',
        'Netherlands': 'Western Europe',
        'Sweden': 'Western Europe',
        
        #Asia
        'Australia': 'Asia-Pacific Developed',
        'Singapore': 'Asia-Pacific Developed',
        'India': 'South Asia',
        
        #Undefinied
        'Remote': 'Remote'
        }
        
        data["location"] = data["location"].map(clusters_of_location)
    
    return data

def convert_ordinal(data: pd.DataFrame) -> pd.DataFrame:
    if "education_level" in data.columns:
        edu_mapping = {
        'High School': 0,
        'Diploma': 1,
        'Bachelor': 2,
        'Master': 3,
        'PhD': 4
        }
        
        data["education_level"] = data["education_level"].map(edu_mapping)
        
    if "company_size" in data.columns:

        size_mapping = {
        'Startup': 0,
        'Small': 1,
        'Medium': 2,
        'Large': 3,
        'Enterprise': 4
        }      
    
        data["company_size"] = data["company_size"].map(size_mapping)
    return data 
    
def convert_ohe(data: pd.DataFrame, features: Tuple) -> pd.DataFrame:
    from sklearn.preprocessing import OneHotEncoder
    
    if len(features) == 0:
        return data
    
    if isinstance(features, pd.Series):
        cat_cols = features.tolist()
    elif isinstance(features, (list, tuple, pd.Index)):
        cat_cols = list(features)

    df_ohe = data.copy()
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
    encoded_array = encoder.fit_transform(df_ohe[cat_cols])

    new_columns = encoder.get_feature_names_out(cat_cols)

    encoded_df = pd.DataFrame(
        encoded_array,
        columns=new_columns,
        index=df_ohe.index
    )

    df_ohe = df_ohe.drop(columns=cat_cols).join(encoded_df)

    return df_ohe
        
def linear_data(drop_features: Tuple = (), num_rows = 5000):
    X, y =  convert_to_batches(num = num_rows)

    if len(drop_features) != 0:
        X = X.drop(columns=drop_features)
        
    X = change_category(X)
    X = convert_ordinal(X)
    
    ohe_features = []
    
    if "job_title" in X.columns:
        ohe_features.append("job_title")
    if "industry" in X.columns:
        ohe_features.append("industry")
    if "location" in X.columns:
        ohe_features.append("location")
    if "remote_work" in X.columns:
        ohe_features.append("remote_work")
        
    X = convert_ohe(X, ohe_features)
    
    return X, y

**2.1 Глупая регрессия**

In [ ]:
def dummy(X: pd.DataFrame, y:pd.Series) -> BaseEstimator:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=47)
    
    param_grid = {"strategy" : ['mean', 'median', 'quantile'],
                  "quantile" : np.linspace(0,1,10)}
    
    dummy_model = GridSearchCV(DummyRegressor(), 
                param_grid=param_grid,
                cv=5)
    dummy_model.fit(X=X_train, y=y_train)
    y_pred = dummy_model.predict(X=X_test)
    
    print_metrics(y_test, y_pred)
    print(dummy_model.best_params_)
    
    return dummy_model.best_estimator_

X, y = linear_data(["job_title", "industry", "location", "remote_work"])

print(f"-------------------Dummy-model-------------------")
dummy_model = dummy(X, y)
print(f"-------------------------------------------------\n")

**2.2 Линейная регрессия**

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

def lasso(X: pd.DataFrame, y:pd.Series, use_best_params: bool = False) -> Tuple[BaseEstimator, pd.DataFrame, pd.Series]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=47)

    if use_best_params == True:
        param_grid = {"alpha" : [4]}
    else:
        param_grid = {"alpha": np.logspace(-6, 6, 13)}
        
    scoring = "neg_mean_squared_error"
    
    Lasso_model = GridSearchCV(estimator=Lasso(max_iter=5000, tol=1e-3),
                            param_grid=param_grid,
                            scoring=scoring,
                            cv = 5)
    Lasso_model.fit(X=X_train, y=y_train)
    
    y_pred = Lasso_model.predict(X=X_test)
    print_metrics(y_test, y_pred)
    print(f"Лучшие параметры: {Lasso_model.best_params_}")
    
    weights = pd.Series(data = Lasso_model.best_estimator_.coef_, index = X.columns, name = "WEIGHTS_L1").sort_values(ascending=False)
    
    return Lasso_model.best_estimator_, X_train, weights

def ridge(X: pd.DataFrame, y:pd.Series, use_best_params: bool = False) -> Tuple[BaseEstimator, pd.DataFrame, pd.Series]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=47)
    
    if use_best_params == True:
        param_grid = {"alpha" : [4]}
    else:
        param_grid = {"alpha": np.logspace(-6, 6, 13)}
        
    scoring = "neg_mean_squared_error"
    
    Ridge_model = GridSearchCV(estimator=Ridge(max_iter=5000, tol=1e-3),
                            param_grid=param_grid,
                            scoring=scoring,
                            cv = 5)
    Ridge_model.fit(X=X_train, y=y_train)
    
    y_pred = Ridge_model.predict(X=X_test)
    print_metrics(y_test, y_pred)
    print(f"Лучшие параметры: {Ridge_model.best_params_}")
    
    weights = pd.Series(data = Ridge_model.best_estimator_.coef_, index = X.columns, name = "WEIGHTS_L2").sort_values(ascending=False)
    
    return Ridge_model.best_estimator_, X_train, weights

def linreg(X: pd.DataFrame, y:pd.Series) -> Tuple[BaseEstimator, pd.DataFrame, pd.Series]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=47)
    Linear_model = cv_fit(X = X_train, y = y_train, model = LinearRegression(), num_folds = 5)
    
    y_pred = Linear_model.predict(X=X_test)
    print_metrics(y_test, y_pred)
    
    weights = pd.Series(data = Linear_model.coef_, index = X.columns, name = "WEIGHTS_L0").sort_values(ascending=False)
    
    return Linear_model, X_train, weights

X, y = linear_data(drop_features = ["industry", "remote_work"], num_rows=5000)

print(f"-------------------Linear-model-------------------")
linear_model, linear_train, l0_weights = linreg(X, y)
print(f"-------------------------------------------------\n")
print(f"-------------------Lasso-model-------------------")
lasso_model, lasso_train, l1_weights = lasso(X,y, use_best_params=True)
print(f"-------------------------------------------------\n")
print(f"-------------------Ridge-model-------------------")
ridge_model, ridge_train, l2_weights = ridge(X,y, use_best_params=True)
print(f"-------------------------------------------------\n")

weights = pd.concat([l0_weights, l1_weights, l2_weights], axis=1).sort_values(by=["WEIGHTS_L0", "WEIGHTS_L1", "WEIGHTS_L2"], ascending=[False, False, False])
weights.Name = "WEIGHTS"
print(weights)



**2.3 Решающее дерево**

In [ ]:
def deep_tree(X:pd.DataFrame, y:pd.Series, use_best_params: bool = False) -> Tuple[BaseEstimator, pd.DataFrame]:
    import time
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=47)
    
    if use_best_params == True:
        param_grid = {"criterion" : ['squared_error'],
                  "splitter" : ["best"],
                  "max_depth" : [12],
                  "min_samples_split" : [15],
                  "min_samples_leaf" : [10],
                  "max_features" : [20]}
    else:
        param_grid = {"criterion" : ['squared_error', 'absolute_error'],
                  "splitter" : ["best", "random"],
                  "max_depth" : [12],
                  "min_samples_split" : [15, 30, 45],
                  "min_samples_leaf" : [5, 10, 15],
                  "max_features" : [5, 15, 20]}
    
    scoring = "neg_mean_squared_error"
    
    deep_tree_model = GridSearchCV(DecisionTreeRegressor(random_state=47),
                              param_grid=param_grid,
                              scoring=scoring, 
                              cv=5)
    
    start_time = time.time()
    
    deep_tree_model.fit(X=X_train, y=y_train)
    
    end_time = time.time()
    print(f"Время обучения: {round(end_time - start_time, 2)} секунд")
    
    y_pred = deep_tree_model.predict(X=X_test)
    print_metrics(y_test, y_pred)
    print(f"Лучшие параметры: {deep_tree_model.best_params_}")
    
    return deep_tree_model.best_estimator_, X_train

def short_tree(X:pd.DataFrame, y:pd.Series, use_best_params: bool = False) -> Tuple[BaseEstimator, pd.DataFrame]:
    import time
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=47)
    
    if use_best_params == True:
        param_grid = {"criterion" : ['absolute_error'],
                  "splitter" : ["best"],
                  "max_depth" : [3],
                  "min_samples_split" : [15],
                  "min_samples_leaf" : [10],
                  "max_features" : [20]}
    else:
        param_grid = {"criterion" : ["squared_error", 'absolute_error'],
                  "splitter" : ["best", "random"],
                  "max_depth" : [3],
                  "min_samples_split" : [15, 30, 45],
                  "min_samples_leaf" : [5],
                  "max_features" : [15]}
    
    
    scoring = "neg_mean_squared_error"
    
    short_tree_model = GridSearchCV(DecisionTreeRegressor(random_state=47),
                              param_grid=param_grid,
                              scoring=scoring, 
                              cv=5)
    
    start_time = time.time()
    
    short_tree_model.fit(X=X_train, y=y_train)
    
    end_time = time.time()
    print(f"Время обучения: {round(end_time - start_time, 2)} секунд")
    
    y_pred = short_tree_model.predict(X=X_test)
    print_metrics(y_test, y_pred)
    print(f"Лучшие параметры: {short_tree_model.best_params_}")
    
    return short_tree_model.best_estimator_, X_train

# X, y = linear_data(num_rows=10000)

# print(f"-------------------Deep-Tree-model-------------------")
# deep_tree_model, deep_tree_train = deep_tree(X, y)
# print(f"-----------------------------------------------------\n")
# print(f"-------------------Short-Tree-model-------------------")
# short_tree_model, short_tree_train = short_tree(X, y)
# print(f"------------------------------------------------------\n")

**2.4 Случайный лес**

In [ ]:
def forest(X:pd.DataFrame, y:pd.Series, use_best_params: bool = False) -> Tuple[BaseEstimator, pd.DataFrame]:
    import time
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=47)
    
    
    
    if use_best_params == True:
        param_grid = {"criterion" : ['squared_error'],
                  "n_estimators" : [600],
                  "max_depth" : [9],
                  "min_samples_split" : [45],
                  "min_samples_leaf" : [16],
                  "max_features" : ["sqrt"]}
    else:
        param_grid = {"criterion" : ['squared_error'],
                  "max_depth" : [7, 8, 9],
                  "min_samples_split" : list(map(int, np.logspace(start=4, stop=8, base=2, num=10))),
                  "min_samples_leaf" : list(map(int,np.logspace(start=3, stop=7, base=2, num=10))),
                  "max_features" : ["sqrt", "log2"]}
    
    
    scoring = "neg_mean_squared_error"
    
    forest_model = GridSearchCV(RandomForestRegressor(random_state=47),
                              param_grid=param_grid,
                              scoring=scoring, 
                              cv=5)
    
    start_time = time.time()
    
    forest_model.fit(X=X_train, y=y_train)
    
    end_time = time.time()
    print(f"Время обучения: {round(end_time - start_time, 2)} секунд")
    
    y_pred = forest_model.predict(X=X_test)
    print_metrics(y_test, y_pred)
    print(f"Лучшие параметры: {forest_model.best_params_}")
    
    return forest_model.best_estimator_, X_train

# X, y = linear_data(num_rows=50000)

# print(f"-------------------Forest-model-------------------")
# forest_model, forest_train = forest(X, y, use_best_params=True)
# print(f"--------------------------------------------------\n")

**2.5 SHAP-Анализ**

In [ ]:
def shap_analysis(model: BaseEstimator, X_dataset: pd.DataFrame, is_tree: bool = False, to_save: bool = False, path_to_dir: str = "D:/Salary-prediction/SHAP", name_graph: str = "SHAP_graph.svg"):
    import shap
    import matplotlib.pyplot as plt

    if is_tree == True:
        explainer = shap.TreeExplainer(model, X_dataset)
    else:
        explainer = shap.LinearExplainer(model, X_dataset)
    
    shap_values = explainer.shap_values(X_dataset)
    
    if to_save == True:
        shap.summary_plot(shap_values, X_dataset, show = False)
        
        plt.savefig(path_to_dir + f"/{name_graph}")
        
        plt.close()
    else:
        shap.summary_plot(shap_values, X_dataset)
        plt.close()
    
X, y = linear_data(num_rows=10000)

print(f"-------------------Ridge-model-------------------")
ridge_model, ridge_train, l2_weights = ridge(X,y, use_best_params=True)
shap_analysis(ridge_model, ridge_train, to_save=True, name_graph="Ridge.svg")   
print(f"-------------------------------------------------\n")

print(f"-------------------Deep-Tree-model-------------------")
deep_tree_model, deep_tree_train = deep_tree(X, y, use_best_params=True)
shap_analysis(deep_tree_model, deep_tree_train, is_tree=True, to_save=True, name_graph="Deep_tree.svg")
print(f"-----------------------------------------------------\n")

print(f"-------------------Short-Tree-model-------------------")
short_tree_model, short_tree_train = short_tree(X, y, use_best_params=True)
shap_analysis(short_tree_model, short_tree_train, is_tree=True, to_save=True, name_graph="Short_tree.svg")
print(f"------------------------------------------------------\n")

print(f"-------------------Forest-model-------------------")
forest_model, forest_train = forest(X, y, use_best_params=True)
shap_analysis(forest_model, forest_train, is_tree=True, to_save=True, name_graph="Forest.svg")
print(f"--------------------------------------------------\n")

# **3.Main модели**

*Загрузка необходимых функций*

In [ ]:
from sklearn.base import BaseEstimator, clone
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
import matplotlib.pyplot as plt
from Logging import Logging
import time, os, json

def print_metrics(y_true: pd.Series, y_pred: pd.Series):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
    
    print(f"MAPE: {round(mean_absolute_percentage_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"MAE: {round(mean_absolute_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"MSE: {round(mean_squared_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"RMSE: {round(root_mean_squared_error(y_true=y_true, y_pred=y_pred),2)}")
    print(f"R2: {round(r2_score(y_true=y_true, y_pred=y_pred),2)}")

def cv_fit(X: pd.DataFrame, y: pd.Series, model: BaseEstimator, num_folds: int = 5) -> BaseEstimator:
    from sklearn.metrics import root_mean_squared_error
    
    folds = KFold(n_splits=num_folds, shuffle=True, random_state=47)
    best_rmse = float('inf')
    best_model = None

    for i, (idx_train, idx_test) in enumerate(folds.split(X)):
        X_train, y_train = X.iloc[idx_train, :], y.iloc[idx_train]
        X_test, y_test = X.iloc[idx_test, :], y.iloc[idx_test]

        model_fold = clone(model)
        model_fold.fit(X_train, y_train)
        y_pred = model_fold.predict(X_test)

        rmse = round(root_mean_squared_error(y_test, y_pred),2)
        if rmse < best_rmse:
            best_rmse = rmse
            best_model = model_fold

    return best_model     

def robust_scale(X:pd.DataFrame, features: Tuple) -> pd.DataFrame:
    from sklearn.preprocessing import RobustScaler
    from sklearn.model_selection import train_test_split
    
    X_scaled = X.copy()
    
    idx_train, idx_test = train_test_split(
        X.index, 
        test_size=0.3, 
        random_state=42  
    )
    
    scaler = RobustScaler()
    
    scaler.fit(X.loc[idx_train, features])
    
    X_scaled.loc[idx_train, features] = scaler.transform(X.loc[idx_train, features])
    X_scaled.loc[idx_test, features] = scaler.transform(X.loc[idx_test, features])
    
    return X_scaled

def change_category(data: pd.DataFrame):    
    if "job_title" in data.columns:
        clusters_of_job = {
        # Data & AI
        'AI Engineer': 'Data & AI',
        'Machine Learning Engineer': 'Data & AI',
        'Data Scientist': 'Data & AI',
        'Data Analyst': 'Data & AI',
        
        # Software Engineering
        'Software Engineer': 'Software Engineering',
        'Backend Developer': 'Software Engineering',
        'Frontend Developer': 'Software Engineering',
        
        # Cloud & DevOps
        'Cloud Engineer': 'Cloud & DevOps',
        'DevOps Engineer': 'Cloud & DevOps',
        
        # Product & Strategy
        'Product Manager': 'Product & Strategy',
        'Business Analyst': 'Product & Strategy',
        
        # Security
        'Cybersecurity Analyst': 'Security'
    }
        
        data["job_title"] = data["job_title"].map(clusters_of_job)
    
    if "location" in data.columns:
        clusters_of_location = { 
        #America
        'USA': 'North America',
        'Canada': 'North America',
        
        #Europe
        'UK': 'Western Europe',
        'Germany': 'Western Europe',
        'Netherlands': 'Western Europe',
        'Sweden': 'Western Europe',
        
        #Asia
        'Australia': 'Asia-Pacific Developed',
        'Singapore': 'Asia-Pacific Developed',
        'India': 'South Asia',
        
        #Undefinied
        'Remote': 'Remote'
        }
        
        data["location"] = data["location"].map(clusters_of_location)
    
    return data

def convert_ordinal(data: pd.DataFrame) -> pd.DataFrame:
    if "education_level" in data.columns:
        edu_mapping = {
        'High School': 0,
        'Diploma': 1,
        'Bachelor': 2,
        'Master': 3,
        'PhD': 4
        }
        
        data["education_level"] = data["education_level"].map(edu_mapping)
        
    if "company_size" in data.columns:

        size_mapping = {
        'Startup': 0,
        'Small': 1,
        'Medium': 2,
        'Large': 3,
        'Enterprise': 4
        }      
    
        data["company_size"] = data["company_size"].map(size_mapping)
    return data 
    
def convert_ohe(data: pd.DataFrame, features: Tuple) -> pd.DataFrame:
    from sklearn.preprocessing import OneHotEncoder
    
    if len(features) == 0:
        return data
    
    if isinstance(features, pd.Series):
        cat_cols = features.tolist()
    elif isinstance(features, (list, tuple, pd.Index)):
        cat_cols = list(features)

    df_ohe = data.copy()
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
    encoded_array = encoder.fit_transform(df_ohe[cat_cols])

    new_columns = encoder.get_feature_names_out(cat_cols)

    encoded_df = pd.DataFrame(
        encoded_array,
        columns=new_columns,
        index=df_ohe.index
    )

    df_ohe = df_ohe.drop(columns=cat_cols).join(encoded_df)

    return df_ohe
        
def linear_data(drop_features: Tuple = (), num_rows = 5000):
    X, y =  convert_to_batches(num = num_rows)

    if len(drop_features) != 0:
        X = X.drop(columns=drop_features)
        
    X = change_category(X)
    X = convert_ordinal(X)
    
    ohe_features = []
    
    if "job_title" in X.columns:
        ohe_features.append("job_title")
    if "industry" in X.columns:
        ohe_features.append("industry")
    if "location" in X.columns:
        ohe_features.append("location")
    if "remote_work" in X.columns:
        ohe_features.append("remote_work")
        
    X = convert_ohe(X, ohe_features)
    
    return X, y


**3.1 CatBoost GBDT**

In [ ]:
from catboost import Pool, CatBoostRegressor

def grid_gbdt(X: pd.DataFrame, y: pd.Series, use_best_params: bool = False, cat_features: List = None, with_eval: bool = True, print_metrics: bool = True) -> Tuple[CatBoostRegressor, Dict]:
    if cat_features is None:
        cat_features = ["job_title", "location", "company_size", "education_level", "industry", "remote_work"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)
    X_train_part, X_val_part, y_train_part, y_val_part = train_test_split(X_train, y_train, test_size=0.2, random_state=47)

    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    test_pool = Pool(X_test, y_test, cat_features=cat_features)
    eval_pool = Pool(X_val_part, y_val_part, cat_features=cat_features)
    
    del X_train, X_test, y_train, X_train_part, X_val_part, y_train_part, y_val_part

    if use_best_params == True: # Лучшие параметры
        param_grid = {
            "n_estimators": [700],
            "max_depth": [3],
            "learning_rate": [0.3]
        }
    else:
        param_grid = {
            "n_estimators": [1000],
            "max_depth": [2, 3],
            "learning_rate": np.linspace(0.1, 0.9, num=5)
        }
        
    model = CatBoostRegressor(random_seed=47, logging_level = 'Silent') 
    
    model.grid_search(
        param_grid, 
        train_pool,
        cv=5,      
        plot=False,          
        verbose=False     
    )
    best_params = model.get_params()
    
    if with_eval == True:
        model = CatBoostRegressor(**best_params)
        model.fit(train_pool, eval_set=eval_pool, verbose=False, plot=False)
    
    dict_metrics = model.eval_metrics(test_pool, metrics=["MAPE", "MAE", "RMSE", "R2"]) 
    if print_metrics == True:
        print(f"MAPE: {round(dict_metrics["MAPE"][-1],2)}")
        print(f"MAE: {round(dict_metrics["MAE"][-1],2)}")
        print(f"MSE: {round(dict_metrics['RMSE'][-1] ** 2,2)}")
        print(f"RMSE: {round(dict_metrics["RMSE"][-1],2)}")
        print(f"R2: {round(dict_metrics["R2"][-1],2)}")
    
    return model, dict_metrics

# X, y = convert_to_batches(load_data(10000, random=True))

# print(f"-------------------GBDT-model-------------------")
# model, metrics = grid_gbdt(X, y, use_best_params=True, with_eval=False)
# print(f"------------------------------------------------\n")

**3.2 Эксперименты с качеством над CatBoost GBDT**

In [ ]:
def exp1_gbdt(X: pd.DataFrame, y: pd.Series) -> CatBoostRegressor: #Проверяем ординальные признаки
    X = X.copy()
    X = convert_ordinal(X)
    X["education_level"] = X["education_level"].astype("int")
    X["company_size"] = X["company_size"].astype("int")
    
    model, metrics = grid_gbdt(X, y, use_best_params=True, cat_features=["job_title", "location", "industry", "remote_work"], with_eval=False)
    
    return model

def exp2_gbdt(X: pd.DataFrame, y: pd.Series) -> CatBoostRegressor: #Проверяем категориальные признаки
    X = X.copy()
    X = change_category(X)
    
    model, metrics = grid_gbdt(X, y, use_best_params=True, with_eval=False)
    
    return model

def exp3_gbdt(X: pd.DataFrame, y: pd.Series) -> CatBoostRegressor: #Объединим exp1 и exp2
    X = X.copy()
    X = convert_ordinal(X)
    X["education_level"] = X["education_level"].astype("int")
    X["company_size"] = X["company_size"].astype("int")
    X = change_category(X)
    
    model, metrics = grid_gbdt(X, y, use_best_params=True, cat_features=["job_title", "location", "industry", "remote_work"], with_eval=False)
    
    return model

def exp4_gbdt(X: pd.DataFrame, y: pd.Series) -> CatBoostRegressor: #Удалим слабые признаки
    X = X.copy()
    X = X.drop(["remote_work", "industry"], axis=1)
    
    model, metrics = grid_gbdt(X, y, use_best_params=True, cat_features=["job_title", "location", "company_size", "education_level"], with_eval=False)
    
    return model

def exp5_gbdt(X: pd.DataFrame, y: pd.Series) -> CatBoostRegressor: #Объединение exp1 и exp4
    X = X.copy()
    X = convert_ordinal(X)
    X["education_level"] = X["education_level"].astype("int")
    X["company_size"] = X["company_size"].astype("int")
    
    X = X.drop(["remote_work", "industry"], axis=1)
    
    model, metrics = grid_gbdt(X, y, use_best_params=True, cat_features=["job_title", "location"], with_eval=False)
    
    return model

X, y = convert_to_batches(num=5000)

print(f"-------------------Exp1-model-------------------")
exp1_gbdt(X,y)
print(f"------------------------------------------------\n")


print(f"-------------------Exp2-model-------------------")
exp2_gbdt(X,y)
print(f"------------------------------------------------\n")


print(f"-------------------Exp3-model-------------------")
exp3_gbdt(X,y)
print(f"------------------------------------------------\n")


print(f"-------------------Exp4-model-------------------")
exp4_gbdt(X,y)
print(f"------------------------------------------------\n")

print(f"-------------------Exp5-model-------------------")
exp5_gbdt(X,y)
print(f"------------------------------------------------\n")

**3.3 Эксперименты с оптимизацией над CatBoost GBDT**

In [ ]:
def exp1_optim(X: pd.DataFrame, y: pd.Series):
    model, metrics = grid_gbdt(X, y, use_best_params=True, with_eval=True)
    
    evals_result = model.get_evals_result()
    train_rmse = evals_result['learn']['RMSE']
    val_rmse = evals_result['validation']['RMSE']
    
    best_iter = model.get_best_iteration()
    if best_iter is None:  # Если use_best_model=False
        best_iter = model.tree_count_

    train_rmse_ = train_rmse[:best_iter]
    val_rmse_ = val_rmse[:best_iter]
    
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1,1,1)
    
    ax.plot(range(1, best_iter + 1), train_rmse_, label='Train RMSE', linewidth=2, color = 'red')
    ax.plot(range(1, best_iter + 1), val_rmse_, label='Validation RMSE', linewidth=2, color = 'green')
    ax.grid(alpha=0.3)
    
    ax.xaxis.set_label_text('Number of trees (n_estimators)', fontsize=12)
    ax.yaxis.set_label_text('RMSE', fontsize=12)
    
    fig.suptitle('CatBoost Learning Curves', fontsize=14)
    fig.legend()
    
    plt.show()

def exp2_optim(max_rows: int = 250000, step: int = 1000):
    def draw_graph(x: List, y: List):
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_subplot(1,1,1)
        
        ax.plot(x, y, label='Train RMSE', linewidth=2, color = 'red')
        ax.scatter(x, y, color = 'purple', s=4)
        ax.grid(alpha=0.3)
        
        ax.xaxis.set_label_text('Number of rows (data)', fontsize=12)
        ax.yaxis.set_label_text('RMSE', fontsize=12)
        
        fig.suptitle('CatBoost Learning Curves', fontsize=14)
        fig.legend()
    
        plt.show()

    logs = Logging()
    
    db = MyDB()
    db.connect_to_db()
    
    rows_x = []
    scores_y = []
    
    for num_rows in range(0, max_rows + step, step):
        if num_rows == 0:
            continue
        
        best_score = float("inf")
        awful_score = 0
        
        query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries 
                        LIMIT {num_rows};"""
        
        df = db.select_query(query=query)               
        df = prepare_dataframe(df = df, in_place=True)
        X, y = convert_to_batches(df=df)
        
        rows_x.append(num_rows)
        
        i = 0
        while i < 2:
            i += 1
            
            start_time = time.time()
            model, metrics = grid_gbdt(X, y, use_best_params=True, with_eval=False, print_metrics=False)
            end_time = time.time()
            
            if metrics['RMSE'][-1] <= best_score:
                awful_score = best_score
                best_score = metrics['RMSE'][-1]
            else:
                awful_score = metrics['RMSE'][-1]
                
            logs.push_log(f"Модель №{int(num_rows/step)} - {i}: Время обучения {int(start_time-end_time)} секунд; RMSE = {round(metrics['RMSE'][-1],2)}")
        
        strange_score = round((2*awful_score + best_score)/3,2) 
        scores_y.append(strange_score)
        
    draw_graph(x=rows_x, y=scores_y)
    
    try:
        os.mkdir('JSONs')
    except FileExistsError:
        pass
    
    try:
        with open("JSONs/Scores.json", 'r', encoding='utf-8') as f:
            data = json.load(f)
            if not isinstance(data, list):
                data = []
    except (FileNotFoundError, json.JSONDecodeError):
        data = []   

    data.append(rows_x)
    data.append(scores_y)
    with open("JSONs/Scores.json", 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    db.close_db()
    
X, y = convert_to_batches(load_data(num = 10000, random=True))

print(f"-------------------Optim1-test-------------------")    
exp1_optim(X, y)
print(f"-------------------------------------------------\n")

print(f"-------------------Optim2-test-------------------") 
exp2_optim(max_rows=100000, step=5000)
print(f"-------------------------------------------------\n")

**3.4 ФИНАЛЬНАЯ МОДЕЛЬ GBDT**

In [ ]:
from catboost import Pool, CatBoostRegressor
from sklearn.model_selection import train_test_split
import pandas as pd
from MyDB import MyDB

def prepare_dataframe(df: pd.DataFrame, in_place: bool = False) -> pd.DataFrame:
    for column in df.columns:
        match (column):
            case "job_title":
                continue
            
            case "experience_years":
                df["experience_years"] = df["experience_years"].astype(dtype = "int", copy = in_place)
                continue
                
            case "education_level":
                df["education_level"] = df["education_level"].astype(dtype = "category", copy = in_place)
                continue
                
            case "skills_count":
                df["skills_count"] = df["skills_count"].astype(dtype = "int", copy = in_place)
                continue
                
            case "industry":
                df["industry"] = df["industry"].astype(dtype = "category", copy = in_place)
                continue
                
            case "company_size":
                df["company_size"] = df["company_size"].astype(dtype = "category", copy = in_place)
                continue
            
            case "location":
                continue
            
            case "remote_work":
                df["remote_work"] = df["remote_work"].astype(dtype = "category", copy = in_place)
                continue
                
            case "certifications":
                df["certifications"] = df["certifications"].astype(dtype = "int", copy = in_place)
                continue
                
            case "salary":
                df["salary"] = df["salary"].astype(dtype = "float", copy = in_place)
                continue
    
    return df

def load_data(query: str = None, num: int = 1000, random: bool = False) -> pd.DataFrame:
    if query is None:
        if num == -1:
            query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries;"""                
            if random == True:
                query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries
                        ORDER BY RANDOM();"""
        else:
            query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries 
                        LIMIT {num};"""
                        
            if random == True:
                query = f"""SELECT job_title, experience_years, education_level, skills_count,
                                    industry, company_size, location, 
                                    remote_work, certifications, salary
                        FROM salaries
                        ORDER BY RANDOM()
                        LIMIT {num};"""
    
    db = MyDB()
    db.connect_to_db()
    
    df = db.select_query(query=query)               
    df = prepare_dataframe(df = df, in_place=True)
    
    db.close_db()
    
    return df

def convert_to_batches(df: pd.DataFrame = None, num: int = 1000) -> Tuple[pd.DataFrame, pd.Series]:
    if df is None:
        df = prepare_dataframe(load_data(num = num))
        
    y = df["salary"]
    
    X = df.drop(["salary"], axis = 1)
    
    return X, y

def convert_ordinal(data: pd.DataFrame) -> pd.DataFrame:
    if "education_level" in data.columns:
        edu_mapping = {
        'High School': 0,
        'Diploma': 1,
        'Bachelor': 2,
        'Master': 3,
        'PhD': 4
        }
        
        data["education_level"] = data["education_level"].map(edu_mapping)
        
    if "company_size" in data.columns:

        size_mapping = {
        'Startup': 0,
        'Small': 1,
        'Medium': 2,
        'Large': 3,
        'Enterprise': 4
        }      
    
        data["company_size"] = data["company_size"].map(size_mapping)
    return data 

def grid_gbdt(X: pd.DataFrame, y: pd.Series, use_best_params: bool = False, cat_features: List = None, with_eval: bool = True, print_metrics: bool = True) -> Tuple[CatBoostRegressor, Dict]:
    if cat_features is None:
        cat_features = ["job_title", "location", "company_size", "education_level", "industry", "remote_work"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=47)
    X_train_part, X_val_part, y_train_part, y_val_part = train_test_split(X_train, y_train, test_size=0.2, random_state=47)

    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    test_pool = Pool(X_test, y_test, cat_features=cat_features)
    eval_pool = Pool(X_val_part, y_val_part, cat_features=cat_features)
    
    del X_train, X_test, y_train, X_train_part, X_val_part, y_train_part, y_val_part

    if use_best_params == True: # Лучшие параметры
        param_grid = {
            "n_estimators": [700],
            "max_depth": [3],
            "learning_rate": [0.3]
        }
    else:
        param_grid = {
            "n_estimators": [1000],
            "max_depth": [2, 3],
            "learning_rate": np.linspace(0.1, 0.9, num=5)
        }
        
    model = CatBoostRegressor(random_seed=47, logging_level = 'Silent') 
    
    model.grid_search(
        param_grid, 
        train_pool,
        cv=5,      
        plot=False,          
        verbose=False     
    )
    best_params = model.get_params()
    
    if with_eval == True:
        model = CatBoostRegressor(**best_params)
        model.fit(train_pool, eval_set=eval_pool, verbose=False, plot=False)
    
    dict_metrics = model.eval_metrics(test_pool, metrics=["MAPE", "MAE", "RMSE", "R2"]) 
    if print_metrics == True:
        print(f"MAPE: {round(dict_metrics["MAPE"][-1],2)}")
        print(f"MAE: {round(dict_metrics["MAE"][-1],2)}")
        print(f"MSE: {round(dict_metrics['RMSE'][-1] ** 2,2)}")
        print(f"RMSE: {round(dict_metrics["RMSE"][-1],2)}")
        print(f"R2: {round(dict_metrics["R2"][-1],2)}")
    
    return model, dict_metrics

def final_model(X: pd.DataFrame, y: pd.Series) -> CatBoostRegressor:
    X = X.copy()
    X = convert_ordinal(X)
    X["education_level"] = X["education_level"].astype("int")
    X["company_size"] = X["company_size"].astype("int")
    
    X = X.drop(["remote_work", "industry"], axis=1)
    
    model, metrics = grid_gbdt(X,y,use_best_params=True, with_eval=False, cat_features=["job_title", "location"])
    
    try:
        os.mkdir('Models')
    except FileExistsError:
        pass
    
    model.save_model(fname = "Models/GDBT_model_final.cbm")
    
    return model

X, y = convert_to_batches(load_data(num = 65000, random=True))

model = final_model(X,y)
